## Laboratorio Virtual N° 2 — Consumo de APIs y Captura de Datos Cambiarios en Vivo

### Paso 1: Configuración: Importación de las librerías especializadas pandas y requests.

In [16]:
import pandas as pd
import requests

### Paso 2: Ingesta Dinámica (API): Consumo del endpoint público de cotizaciones cambiarias mediante el método requests.get().

In [17]:
url_base = "https://dolarapi.com"
punto_final = "/v1/dolares" # Endpoint corregido con la información del usuario

respuesta = requests.get(f"{url_base}{punto_final}")

if respuesta.status_code == 200:
    datos = respuesta.json()
    print("Llamada a la API exitosa. Datos recibidos:")
    # Mostrar los primeros elementos para inspeccionar la estructura
    print(datos[:2])
else:
    print(f"Error: {respuesta.status_code} - {respuesta.text}")
    datos = [] # Inicializar lista vacía para evitar errores posteriores

Llamada a la API exitosa. Datos recibidos:
[{'moneda': 'USD', 'casa': 'oficial', 'nombre': 'Oficial', 'compra': 1460, 'venta': 1510, 'fechaActualizacion': '2026-08-14T18:55:00.000Z'}, {'moneda': 'USD', 'casa': 'blue', 'nombre': 'Blue', 'compra': 1525, 'venta': 1545, 'fechaActualizacion': '2026-08-15T14:58:00.000Z'}]


### Paso 3: Estructuración: Transformación del objeto JSON crudo en un DataFrame bidimensional de Pandas.

In [28]:
if datos:
    df_dolares = pd.DataFrame(datos)
    print("DataFrame creado exitosamente:")
    display(df_dolares.head())
    print("Información del DataFrame:")
    df_dolares.info()
else:
    print("No hay datos para crear el DataFrame. Por favor, revisa la llamada a la API en el Paso 2.")
    df_dolares = pd.DataFrame() # Crear un DataFrame vacío



DataFrame creado exitosamente:


,moneda,casa,nombre,compra,venta,fechaActualizacion
0,USD,oficial,Oficial,1460.0,1510.0,2026-08-14T18:55:00.000Z
1,USD,blue,Blue,1525.0,1545.0,2026-08-15T14:58:00.000Z
2,USD,bolsa,Bolsa,1509.6,1521.6,2026-08-15T14:58:00.000Z
3,USD,contadoconliqui,Contado con liquidación,1571.1,1573.8,2026-08-15T14:58:00.000Z
4,USD,mayorista,Mayorista,1478.5,1487.5,2026-08-14T16:12:00.000Z


Información del DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   moneda              7 non-null      object 
 1   casa                7 non-null      object 
 2   nombre              7 non-null      object 
 3   compra              7 non-null      float64
 4   venta               7 non-null      float64
 5   fechaActualizacion  7 non-null      object 
dtypes: float64(2), object(4)
memory usage: 468.0+ bytes


### Paso 4: Limpieza y Munging de Datos: Selección de las variables de interés (casa, compra, venta, fechaActualizacion), renombrado de columnas a un estándar profesional y conversión del campo de fecha a tipo datetime para asegurar la calidad temporal de la información.

In [19]:
if not df_dolares.empty:
    # Seleccionar las variables de interés
    # Nota: 'casa' generalmente se refiere a 'nombre', 'compra', 'venta' y 'fechaActualizacion' son directas.
    # Ajusta 'casa' si la estructura JSON usa una clave diferente para el tipo de dólar (ej., 'nombre' o 'casa')
    # Basado en la salida de print(datos[:2]), 'nombre' parece ser el equivalente a 'casa'.
    columnas_a_mantener = ['nombre', 'compra', 'venta', 'fechaActualizacion']

    # Verificar si todas las columnas existen antes de continuar
    columnas_faltantes = [col for col in columnas_a_mantener if col not in df_dolares.columns]
    if columnas_faltantes:
        print(f"Advertencia: Las siguientes columnas esperadas faltan en el DataFrame: {columnas_faltantes}")
        print("Las columnas disponibles son:", df_dolares.columns.tolist())
        # Intentar continuar con las columnas disponibles, o manejar el error
        df_limpio = df_dolares[[col for col in columnas_a_mantener if col in df_dolares.columns]].copy()
    else:
        df_limpio = df_dolares[columnas_a_mantener].copy()

    # Renombrar columnas a un estándar profesional
    df_limpio = df_limpio.rename(columns={
        'nombre': 'tipo_cotizacion',
        'compra': 'valor_compra',
        'venta': 'valor_venta',
        'fechaActualizacion': 'fecha_actualizacion'
    })

    # Convertir 'fecha_actualizacion' a tipo datetime
    # Usar errors='coerce' para convertir los valores inválidos a NaT (Not a Time) sin generar un error
    df_limpio['fecha_actualizacion'] = pd.to_datetime(df_limpio['fecha_actualizacion'], errors='coerce')

    print("DataFrame limpio:")
    display(df_limpio.head())
    print("Información del DataFrame después de la limpieza:")
    df_limpio.info()
else:
    print("El DataFrame está vacío, se omite el paso de limpieza.")

DataFrame limpio:


,tipo_cotizacion,valor_compra,valor_venta,fecha_actualizacion
0,Oficial,1460.0,1510.0,2026-08-14 18:55:00+00:00
1,Blue,1525.0,1545.0,2026-08-15 14:58:00+00:00
2,Bolsa,1509.6,1521.6,2026-08-15 14:58:00+00:00
3,Contado con liquidación,1571.1,1573.8,2026-08-15 14:58:00+00:00
4,Mayorista,1478.5,1487.5,2026-08-14 16:12:00+00:00


Información del DataFrame después de la limpieza:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   tipo_cotizacion      7 non-null      object             
 1   valor_compra         7 non-null      float64            
 2   valor_venta          7 non-null      float64            
 3   fecha_actualizacion  7 non-null      datetime64[ns, UTC]
dtypes: datetime64[ns, UTC](1), float64(2), object(1)
memory usage: 356.0+ bytes


### Paso 5: Análisis de Negocio: Cálculo automatizado de la brecha cambiaria (spread) porcentual por tipo de cotización y determinación de las métricas de tendencia central (media) y dispersión (desvío estándar) del mercado cambiario.

In [20]:
if not df_limpio.empty:
    # Calcular la brecha porcentual (spread) para cada tipo de cotización
    # Brecha = ((Venta - Compra) / Compra) * 100
    df_limpio['brecha_porcentual'] = ((df_limpio['valor_venta'] - df_limpio['valor_compra']) / df_limpio['valor_compra']) * 100

    print("DataFrame con brecha calculada:")
    display(df_limpio.head())

    print("\n--- Métricas de Tendencia Central y Dispersión (por tipo de cotización) ---")
    # Determinar la tendencia central (media) y la dispersión (desvío estándar) para 'valor_compra', 'valor_venta' y 'brecha_porcentual'
    metricas_analisis = df_limpio.groupby('tipo_cotizacion')[['valor_compra', 'valor_venta', 'brecha_porcentual']].agg(['mean', 'std'])
    display(metricas_analisis)

    print("\n--- Resumen General de Métricas ---")
    print("Media (promedio) de los valores de compra:", df_limpio['valor_compra'].mean())
    print("Desvío estándar de los valores de compra:", df_limpio['valor_compra'].std())
    print("Media (promedio) de los valores de venta:", df_limpio['valor_venta'].mean())
    print("Desvío estándar de los valores de venta:", df_limpio['valor_venta'].std())
    print("Media (promedio) de la brecha porcentual:", df_limpio['brecha_porcentual'].mean())
    print("Desvío estándar de la brecha porcentual:", df_limpio['brecha_porcentual'].std())

else:
    print("El DataFrame está vacío, se omite el paso de análisis.")

DataFrame con brecha calculada:


,tipo_cotizacion,valor_compra,valor_venta,fecha_actualizacion,brecha_porcentual
0,Oficial,1460.0,1510.0,2026-08-14 18:55:00+00:00,3.424658
1,Blue,1525.0,1545.0,2026-08-15 14:58:00+00:00,1.311475
2,Bolsa,1509.6,1521.6,2026-08-15 14:58:00+00:00,0.794913
3,Contado con liquidación,1571.1,1573.8,2026-08-15 14:58:00+00:00,0.171854
4,Mayorista,1478.5,1487.5,2026-08-14 16:12:00+00:00,0.608725



--- Métricas de Tendencia Central y Dispersión (por tipo de cotización) ---


valor_compra     valor_venta     brecha_porcentual    
                                mean std        mean std              mean std
tipo_cotizacion                                                               
Blue                         1525.00 NaN     1545.00 NaN          1.311475 NaN
Bolsa                        1509.60 NaN     1521.60 NaN          0.794913 NaN
Contado con liquidación      1571.10 NaN     1573.80 NaN          0.171854 NaN
Cripto                       1572.13 NaN     1577.48 NaN          0.340303 NaN
Mayorista                    1478.50 NaN     1487.50 NaN          0.608725 NaN
Oficial                      1460.00 NaN     1510.00 NaN          3.424658 NaN
Tarjeta                      1898.00 NaN     1963.00 NaN          3.424658 NaN


--- Resumen General de Métricas ---
Media (promedio) de los valores de compra: 1573.4757142857145
Desvío estándar de los valores de compra: 149.24844654212686
Media (promedio) de los valores de venta: 1596.9114285714288
Desvío estándar de los valores de venta: 164.7339397764945
Media (promedio) de la brecha porcentual: 1.4395121226868937
Desvío estándar de la brecha porcentual: 1.403503476312107


### Cuestionario Obligatorio de Reflexión de Negocio

A continuación, deberán responder las siguientes tres preguntas en una celda de texto (Markdown) al final de su entrega:

1.  **Análisis del Fenómeno:** Expliquen en tres oraciones cómo influye la existencia de múltiples cotizaciones (dólar oficial, MEP, Blue, tarjeta) en la previsibilidad de costos de una empresa importadora vs. una exportadora en Argentina.

    *Respuesta:* Al existir múltiples cotizaciones de una misma moneda, la previsibilidad de costos de las empresas se ve perjudicado ya que añade mayor incertidumbre al cálculo de estos. Esto es mayoritariamente visible en las empresas importadoras de bienes y servicios, las cuales deberán considerar la brecha entre las cotizaciones dentro de su planificación de costos, aumentando el riesgo de proyectos y afectando sus decisiones, haciendolas más difíciles. Para las empresas exportadoras, estas diferencias en las cotizaciones pueden afectar sus márgenes de ganancia, para bien o para mal dependiendo si el dólar oficial (el dólar al que las empresas deben apegarse para las operaciones en el país) está por encima o por debajo del valor real del peso, lo cual puede, de nuevo, afectar la planificación de la empresa entorno a sus costos.

2.  **Calidad de Datos:** Al revisar los datos en crudo que devolvió el servidor, ¿qué inconsistencia de huso horario o formato observaron en la columna `fechaActualizacion` antes de realizar la conversión con `pd.to_datetime()`?

    *Respuesta:* La columna `fechaActualizacion` utiliza `object` como formato en lugar de `datatime64`.

3.  **Métrica de Decisión:** Si fueran el CFO de una corporación y tuvieran que liquidar divisas para pagar sueldos mañana, ¿por qué basar la decisión únicamente en el "promedio" de las cotizaciones de la API sería un error crítico de tesorería? Justifiquen utilizando los conceptos de mínimo, máximo y desvío estándar obtenidos en su análisis.

    *Respuesta:* Basarse únicamente en el promedio ignora el dato de que la cotización del dólar (y más en este país) es volátil. La desviación estándar nos muestra esa volatilidad, sumado a los máximos y mínimos, es posible ver la distribución completa de los valores. De no usar todas las variables para, en este caso, pagar sueldos, se estaría asumiendo un riesgo muy alto en la necesidad de caja de la empresa para pagar esos sueldos.